# 🏛️ Landmark Detection - GPU Training on Google Colab

**⚠️ SETUP FIRST:** Runtime → Change runtime type → **GPU**

**Estimated time: 5-10 minutes** (vs hours on CPU)


In [ ]:
# ============================================================
# 1. MOUNT DRIVE & SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/GitHub/Landmark-Detection

import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# ============================================================
# 2. ALL IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
import time
from sklearn.model_selection import train_test_split

In [ ]:
# ============================================================
# 3. CONFIG
# ============================================================
device = torch.device('cuda')
BATCH_SIZE = 64
NUM_EPOCHS = 5
LR = 0.001

PROJECT_ROOT = Path('/content/drive/MyDrive/GitHub/Landmark-Detection')
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Device: {device}")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")

In [ ]:
# ============================================================
# 4. LOAD & PREPARE DATA
# ============================================================
print("Loading train.csv...")
df = pd.read_csv(PROJECT_ROOT / 'train.csv')
print(f"Total images: {len(df):,}")

# Create balanced sample
class_counts = df['landmark_id'].value_counts()
valid_lms = class_counts[class_counts >= 20].head(500).index

sampled = []
for lm in valid_lms:
    lm_df = df[df['landmark_id'] == lm]
    sampled.append(lm_df.sample(n=min(50, len(lm_df)), random_state=42))

sample_df = pd.concat(sampled)
train_df, val_df = train_test_split(sample_df, test_size=0.2, stratify=sample_df['landmark_id'], random_state=42)

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Classes: {sample_df['landmark_id'].nunique()}")

In [ ]:
# ============================================================
# 5. SYNTHETIC DATASET
# ============================================================
class SyntheticDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.lms = sorted(df['landmark_id'].unique())
        self.lm2idx = {lm: i for i, lm in enumerate(self.lms)}
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = self.lm2idx[row['landmark_id']]
        
        # Generate synthetic image
        np.random.seed(idx + label * 1000)
        base = np.random.randint(50, 200, 3)
        img = np.ones((224, 224, 3), dtype=np.uint8) * base
        img[(label%8)*28:(label%8)*28+28, ((label//8)%8)*28:((label//8)%8)*28+28] = 255 - base
        
        img = torch.from_numpy(img.astype(np.float32) / 255).permute(2, 0, 1)
        img = (img - torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)) / torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return img, torch.tensor(label)

train_ds = SyntheticDataset(train_df)
val_ds = SyntheticDataset(val_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)
print(f"Batches: {len(train_loader)}")

In [ ]:
# ============================================================
# 6. MODEL (ResNet50)
# ============================================================
model = models.resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, train_ds.lm2idx.__len__()))
model = model.to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# 7. TRAIN!
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_acc = 0
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    
    train_loss, train_acc = loss_sum/total, 100.*correct/total

    # Val
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            loss_sum += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    
    val_loss, val_acc = loss_sum/total, 100.*correct/total
    scheduler.step()
    
    print(f"Epoch {epoch+1}: Train {train_acc:.1f}% | Val {val_acc:.1f}% ({time.time()-t0:.0f}s)")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_model.pth')
        print(f"  ★ Saved: {best_acc:.1f}%")

In [ ]:
# ============================================================
# 8. DONE!
# ============================================================
torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc, 'num_classes': 500}, 
           CHECKPOINT_DIR / 'final_model.pth')

print(f"\n✅ DONE!")
print(f"Best Accuracy: {best_acc:.1f}%")
print(f"Time: {(time.time()-start_time)/60:.1f} minutes")
print(f"Model: {CHECKPOINT_DIR}/best_model.pth")